#  Imagenette Image Classification & Grad-CAM Workshop
From Image Classification to Explainable AI
Duration: approximately 45 minutes 
In this hands-on workshop, you will use PyTorch to adapt a ResNet18 image classifier to the Imagenette dataset, train it, inspect its predictions, and use Gradient-weighted Class Activation Mapping (Grad-CAM) to visualize image regions that contribute to a prediction.
 
You will complete short code sections by filling in missing expressions and arguments. Run the cells in order.
# Learning objectives
By the end of this workshop, you should be able to:
1. Load and explore an image classification dataset.
2. Explain the role of a pretrained CNN and its classification layer.
3. Configure and train a model using PyTorch.
4. Inspect predictions and confidence scores.
5. Generate and interpret a Grad-CAM heatmap.
 
Before the workshop: Install PyTorch and Torchvision using the official PyTorch installation selector for your computer (CPU or CUDA). Dataset download and training time depend on your internet connection and hardware.


# Section 1: Environment Setup

 We will use:
 - PyTorch / Torchvision for the model, training, and image processing.
 - Matplotlib to display images and plots.
 - pytorch-grad-cam to generate visual explanations.
 
 The PyTorch installation is intentionally separate because the correct command depends on your operating system and CUDA setup.



In [ ]:

pip install grad-cam
pip install matplotlib
pip install numpy
pip install pillow
pip install tqdm
pip install scikit-learn

In [ ]:
## Install PyTorch and torchvision

import os
import tarfile
import urllib.request
import random
import copy

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Make the small workshop sample reproducible
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

## Section 2: Download and Explore Imagenette

Imagenette is a smaller, 10-class subset of ImageNet. Its categories include objects such as a church, golf ball, garbage truck, and parachute.
 
We will use the 160-pixel version to keep the download manageable. The notebook downloads and extracts the dataset only if it is not already present.

In [ ]:
DATA_URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"
DATA_DIR = "imagenette2-160"
ARCHIVE = "imagenette2-160.tgz"

if not os.path.isdir(DATA_DIR):
    print("Downloading Imagenette...")
    urllib.request.urlretrieve(DATA_URL, ARCHIVE)

    print("Extracting dataset...")
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        tar.extractall()

    os.remove(ARCHIVE)

print("Dataset ready:", os.path.abspath(DATA_DIR))

## Prepare the images
A neural network expects tensors of consistent dimensions. The pretrained ResNet18 model also expects images normalized with the ImageNet mean and standard deviation.
 
We use separate transformations for training and validation. Training includes a small random horizontal flip; validation stays deterministic.

In [ ]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
val_dataset = ImageFolder(os.path.join(DATA_DIR, "val"), transform=val_transform)

print("Classes:", train_dataset.classes)
print("Number of classes:", len(train_dataset.classes))
print("Training images available:", len(train_dataset))
print("Validation images available:", len(val_dataset))

# Ensure the class-to-index mapping agrees

assert train_dataset.class_to_idx == val_dataset.class_to_idx

## Exercise 1: Inspect an image
The dataset stores images as tensors after applying the transformations. To display one, we reverse the normalization.
 
Look at the images and answer:
What visual differences do you notice between the ten categories? Why might some categories be harder to distinguish than others?

In [ ]:
def show_tensor_image(image_tensor, title):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image = image_tensor.cpu() * std + mean
    image = image.clamp(0, 1).permute(1, 2, 0).numpy()

    plt.imshow(image)
    plt.title(title)
    plt.axis("off")

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    image, label = train_dataset[i]
    plt.sca(ax)
    show_tensor_image(image, train_dataset.classes[label])
plt.tight_layout()
plt.show()

## Create small, balanced workshop subsets
Training on the entire dataset may take too long for a 45-minute session, especially on a CPU. We will select a small, balanced sample from each class for the live exercise. You can increase these numbers after the workshop.
 
A DataLoader groups examples into batches and shuffles the training examples.

In [ ]:
def balanced_indices(dataset, samples_per_class):
    selected = []
    targets = np.array(dataset.targets)

    for class_id in range(len(dataset.classes)):
        class_indices = np.where(targets == class_id)[0].tolist()
        random.shuffle(class_indices)
        selected.extend(class_indices[:samples_per_class])

    return selected

TRAIN_PER_CLASS = 100
VAL_PER_CLASS = 30

train_indices = balanced_indices(train_dataset, TRAIN_PER_CLASS)
val_indices = balanced_indices(val_dataset, VAL_PER_CLASS)

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)

BATCH_SIZE = 32

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Training examples:", len(train_subset))
print("Validation examples:", len(val_subset))
print("Training batches per epoch:", len(train_loader))

## Section 3: Build the Image Classifier
We will use ResNet18, a convolutional neural network with residual connections. We start with ImageNet-pretrained weights, which provide useful visual features learned from a large image dataset.
 
The original ResNet18 classifier produces 1,000 outputs. Imagenette has 10 classes, so we must replace the final fully connected layer.
 
## Exercise 2: Adapt the final layer
 Fill in the two blanks:
- The input feature count is stored in the existing fully connected layer.
- The new layer must produce one score for each Imagenette class.

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Freeze most pretrained layers to make workshop training faster.
for parameter in model.parameters():
    parameter.requires_grad = False

# Keep the final ResNet block trainable so Grad-CAM can use its gradients.
for parameter in model.layer4.parameters():
    parameter.requires_grad = True

# Find the number of input features
num_features = model.fc.________

# Replace the final layer with one output per Imagenette class
model.fc = nn.Linear(num_features, ________)

model = model.to(device)

print(model.fc)


Only the new final layer is trainable in this short workshop. This is called feature extraction: the pretrained network provides image features, and we train a new classifier for our ten categories.
 
For a longer experiment, you could unfreeze some or all of the pretrained layers and fine-tune them.


## Configure training

An epoch is one pass through the training dataset. The learning rate controls the size of the optimizer's updates. We use cross-entropy loss for this multi-class classification task.
### Exercise 3: Choose the training settings

Start with the suggested values. If there is time, change the learning rate and discuss what you expect to happen.

In [ ]:
EPOCHS = 2
LEARNING_RATE = 0.001

criterion = nn.CrossEntropyLoss()

# Only optimize the trainable final layer
optimizer = optim.Adam(
    (parameter for parameter in model.parameters() if parameter.requires_grad),
    lr=LEARNING_RATE
)


## Train the model
Each training batch follows five steps:
1. Clear gradients from the previous update.
2. Run a forward pass to get model outputs.
3. Calculate the loss.
4. Backpropagate to calculate gradients.
5. Update the trainable weights.
 
## Exercise 4: Complete the training loop
Fill in the missing methods and variables. The code reports average training loss after each epoch.

In [ ]:
history = {"train_loss": []}

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"):
        images = images.to(device)
        labels = labels.to(device)

        1. Clear previous gradients
        optimizer.________

        2. Forward pass
        outputs = model(________)

        3. Calculate loss
        loss = criterion(________, labels)

        4. Backpropagation
        loss.________

        5. Update model weights
        optimizer.________

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    history["train_loss"].append(epoch_loss)
    print(f"Epoch {epoch + 1}: training loss = {epoch_loss:.4f}")

In [ ]:
plt.plot(range(1, EPOCHS + 1), history["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Training Loss")
plt.grid(True)
plt.show()


## Section 4: Evaluate the Model
 Training loss tells us how well the model fits the training examples. We also want to measure performance on validation images that were not used for training.

 The validation loop does not update model weights, so we disable gradient calculation here.

In [ ]:
model.eval()
correct = 0
total = 0
all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

accuracy = 100 * correct / total
print(f"Validation accuracy: {accuracy:.2f}%")

## Inspect one prediction
A confidence score is the model's softmax probability for a class. It is not a guarantee that the prediction is correct.

In [ ]:

# select a validation example
image, true_label = val_dataset[0]
input_tensor = image.unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model(input_tensor)
    probabilities = torch.softmax(output, dim=1)

predicted_class = output.argmax(dim=1).item()
confidence = probabilities[0, predicted_class].item()

print("Actual class:", val_dataset.classes[true_label])
print("Predicted class:", val_dataset.classes[predicted_class])
print(f"Confidence: {confidence:.1%}")

plt.figure(figsize=(4, 4))
show_tensor_image(image, f"Prediction: {val_dataset.classes[predicted_class]}")
plt.show()


## Section 5: Explain a Prediction with Grad-CAM
 Grad-CAM uses gradients flowing into a convolutional layer to produce a coarse heatmap. The heatmap highlights image regions that contribute positively to the score for a selected class.

It is an explanation technique, not another classifier. It does not prove that the model understands the object or that its prediction is correct.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

## Exercise 5: Select the target layer
For ResNet18, we will use the final residual block in layer4. This layer contains high-level visual features and is a common target for Grad-CAM.

In [ ]:
target_layers = [model.________]


## Generate the heatmap
We will explain the model's predicted class for the same validation image. Grad-CAM needs gradients, so do not wrap the CAM call in torch.no_grad().

In [ ]:
# Tell Grad-CAM which class score to explain
targets = [ClassifierOutputTarget(predicted_class)]

with GradCAM(model=model, target_layers=target_layers) as cam:
    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=targets
    )[0]

# Reverse normalization for a display-ready RGB image
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

original_image = image.permute(1, 2, 0).cpu().numpy()
original_image = np.clip(original_image * std + mean, 0, 1)

visualization = show_cam_on_image(
    original_image.astype(np.float32),
    grayscale_cam,
    use_rgb=True
)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(original_image)
axes[0].set_title("Original image")
axes[0].axis("off")

axes[1].imshow(visualization)
axes[1].set_title(f"Grad-CAM: {val_dataset.classes[predicted_class]}")
axes[1].axis("off")

plt.tight_layout()
plt.show()